# Component 3 - end-to-end pipeline

One image in -> ROI (EfficientNet-B0) -> StarDist cells -> complete cells -> biomarkers (0.7960 um/px)
-> JSON report + per-cell CSV + overlay + quality flags (`pipeline.py`), plus the XAI step 2 explanations:
95 % CIs, per-cell reliability (green / yellow / red) and the cells driving CV (they never change a number). Same models and rules as the
reported Phase 6 results; cell 3 checks that the numbers are identical.

**Runtime > Change runtime type > T4 or L4 GPU.**

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" checkout -- ml/cellular/notebooks/
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Packages (Colab only) + unit tests
!pip -q install tensorflow tf-keras stardist
!python -m pytest tests/test_pipeline.py tests/test_roi.py tests/test_biomarkers.py tests/test_xai_biomarkers.py -q

In [ ]:
# 3. Reproducibility: pipeline vs the Phase 6 batch results on 10 validation images
!python -m pipeline --check 10 2>&1 | grep -v -E "Warning|warn|oneDNN"

In [ ]:
# 4. Analyse three images (one per ECD band): overlay, reliability, CV evidence
import pandas as pd
from config import load_config
IMAGES_DIR = load_config()["paths"]["images_dir"]
from IPython.display import Image, display
man = pd.read_csv("splits/manifest.csv")
val = man[(man["split"] == "val") & (man["status"] == "included")]
pick = [g.sample(1, random_state=1).iloc[0] for _, g in val.groupby("ecd_band")]
FILES = " ".join('"' + str(IMAGES_DIR) + "/" + r["image_file"] + '"' for r in pick)
!python -m pipeline {FILES} 2>&1 | grep -v -E "Warning|warn|oneDNN"
for r in pick:
    print(r["image_id"], "- Konan: ECD", r["ecd"], " CV", r["cv"], " HEX", r["hex"])
    stem = r["image_file"].rsplit(".", 1)[0]
    for kind in ("overlay", "reliability", "cv_evidence"):
        display(Image(f"{OUTPUT_ROOT}/pipeline/{stem}_{kind}.png", width=1000))

In [ ]:
# 5. One full report (what the platform receives: now with uncertainty / reliability / evidence)
#    and the start of its per-cell table (now with confidence and level)
r = pick[0]; stem = r["image_file"].rsplit(".", 1)[0]
print(open(f"{OUTPUT_ROOT}/pipeline/{stem}.json").read())
pd.read_csv(f"{OUTPUT_ROOT}/pipeline/{stem}_cells.csv").head()